<a href="https://colab.research.google.com/github/felimaker/IBM-Data-Science-Capstone/blob/main/3_data_wrangling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 3: Metodología de Depuración de Datos (Data Wrangling)
**Proyecto:** Predicción de aterrizaje de la primera etapa del Falcon 9

**Objetivo:** Limpiar el dataset obtenido de la API de SpaceX, tratar valores nulos,
y construir la **variable objetivo `Class`** (1 = aterrizaje exitoso de la primera
etapa, 0 = fallido) a partir de la columna `Outcome`.

**Diagrama de flujo:**

```
dataset_part_1.csv (Notebook 1)
        │
        ▼
Explorar valores nulos por columna (df.isnull().sum())
        │
        ▼
Analizar frecuencia de LaunchSite, Orbit y Outcome
        │
        ▼
Definir conjunto de "bad outcomes" (aterrizajes fallidos)
        │
        ▼
Crear columna binaria Class (1 = éxito, 0 = fallo) mediante landing_class()
        │
        ▼
Imputar PayloadMass nulo con la media de la columna
        │
        ▼
Exportar dataset_part_2.csv (dataset limpio, listo para EDA)
```


In [2]:
import pandas as pd
import numpy as np

# Cargamos el archivo directamente desde la URL del curso
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv"
df = pd.read_csv(url)

df.head()


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


## 1. Identificar y cuantificar valores nulos

In [3]:
df.isnull().sum() / len(df) * 100


,0
FlightNumber,0.000000
Date,0.000000
BoosterVersion,0.000000
PayloadMass,0.000000
Orbit,0.000000
LaunchSite,0.000000
Outcome,0.000000
Flights,0.000000
GridFins,0.000000
Reused,0.000000


In [4]:
df.dtypes


,0
FlightNumber,int64
Date,object
BoosterVersion,object
PayloadMass,float64
Orbit,object
LaunchSite,object
Outcome,object
Flights,int64
GridFins,bool
Reused,bool


## 2. Análisis de frecuencia por columna categórica

In [5]:
# Número de lanzamientos por sitio
print(df['LaunchSite'].value_counts())


LaunchSite
CCAFS SLC 40    55
KSC LC 39A      22
VAFB SLC 4E     13
Name: count, dtype: int64


In [7]:
# Número y ocurrencia de cada órbita
print(df['Orbit'].value_counts())


Orbit
GTO      27
ISS      21
VLEO     14
PO        9
LEO       7
SSO       5
MEO       3
HEO       1
ES-L1     1
SO        1
GEO       1
Name: count, dtype: int64


In [6]:
# Número de ocurrencias de cada resultado de aterrizaje (Outcome)
landing_outcomes = df['Outcome'].value_counts()
landing_outcomes


,count
Outcome,
True ASDS,41
None None,19
True RTLS,14
False ASDS,6
True Ocean,5
False Ocean,2
None ASDS,2
False RTLS,1


## 3. Definir la variable objetivo `Class`

Algunos `Outcome` representan aterrizajes fallidos (o sin intento de aterrizaje);
los agrupamos como "malos resultados" (`bad_outcome = 0`), y todo lo demás lo
consideramos aterrizaje exitoso (`1`).


In [8]:
for i, outcome in enumerate(landing_outcomes.keys()):
    print(i, outcome)


0 True ASDS
1 None None
2 True RTLS
3 False ASDS
4 True Ocean
5 False Ocean
6 None ASDS
7 False RTLS


In [10]:
bad_outcomes = set(landing_outcomes.keys()[[1, 3, 5, 6, 7]])
bad_outcomes


{'False ASDS', 'False Ocean', 'False RTLS', 'None ASDS', 'None None'}

In [11]:
landing_class = [0 if outcome in bad_outcomes else 1 for outcome in df['Outcome']]
df['Class'] = landing_class
df[['Outcome', 'Class']].head(10)


,Outcome,Class
0,None None,0
1,None None,0
2,None None,0
3,False Ocean,0
4,None None,0
5,None None,0
6,True Ocean,1
7,True Ocean,1
8,None None,0
9,None None,0


In [12]:
success_rate = df['Class'].mean()
print(f"Tasa de éxito de aterrizaje global: {success_rate:.2%}")


Tasa de éxito de aterrizaje global: 66.67%


## 4. Imputar valores nulos de `PayloadMass`

In [13]:
mean_payload = df['PayloadMass'].mean()
df['PayloadMass'].replace(np.nan, mean_payload, inplace=True)

print("Nulos restantes en PayloadMass:", df['PayloadMass'].isnull().sum())


Nulos restantes en PayloadMass: 0


/tmp/ipykernel_654/545975869.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['PayloadMass'].replace(np.nan, mean_payload, inplace=True)


## 5. Exportar dataset limpio

In [15]:
df.to_csv('dataset_part_2.csv', index=False)
print(df.shape)
df.head()


(90, 18)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,0


## Resumen de la metodología
- Se identificaron los porcentajes de valores nulos por columna; solo `PayloadMass`
  y `LandingPad` presentaban nulos relevantes.
- Se construyó la variable objetivo binaria `Class` a partir de los 8 valores únicos
  de `Outcome`, agrupando resultados fallidos/sin intento como `0` y aterrizajes
  exitosos como `1`.
- Se imputó `PayloadMass` con la media de la columna (estrategia simple y
  suficiente dado el bajo porcentaje de nulos).
- `LandingPad` se deja con nulos intencionalmente (nulo = no hubo intento de
  aterrizaje en una plataforma, lo cual es información válida en sí misma).
- Resultado: `dataset_part_2.csv`, listo para EDA con visualización y SQL.
- Repositorio de GitHub: **`https://github.com/felimaker/IBM-Data-Science-Capstone`**
